# 18. 3D geometry and rendering — NeRF and complete small 3D Gaussian rasterization

The scene/image sizes are reduced, but projection, screen covariance, view-dependent color, rasterization and compositing are all executed.


In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. Camera projection


In [ ]:
fx = fy = 40.0
cx = cy = 8.0
R_camera = torch.eye(3, device=device)
t_camera = torch.zeros(3, device=device)

def project_points(points):
    camera = points @ R_camera.T + t_camera
    u = fx * camera[:, 0] / camera[:, 2] + cx
    v = fy * camera[:, 1] / camera[:, 2] + cy
    return camera, torch.stack([u, v], dim=-1)

points = torch.tensor([[0.0, 0.0, 2.0], [0.2, -0.1, 2.5]], device=device)
print("pixels:", project_points(points)[1])


## 2. NeRF ray volume rendering


In [ ]:
depths = torch.linspace(0.5, 3.0, 6, device=device)
density = torch.tensor([0.2, 0.5, 1.0, 0.3, 0.1, 0.05], device=device)
colors = torch.rand(6, 3, device=device)
delta = torch.diff(depths)
delta = torch.cat([delta, delta[-1:]])
alpha = 1 - torch.exp(-density * delta)
survival = torch.cat([torch.ones(1, device=device), 1 - alpha + 1e-8])
transmittance = torch.cumprod(survival, dim=0)[:-1]
weights = transmittance * alpha
pixel_color = (weights[:, None] * colors).sum(dim=0)
print("NeRF color:", pixel_color)


## 3. 3D covariance and screen-space projection


In [ ]:
def covariance_3d(scales, angle):
    c = torch.cos(angle)
    s = torch.sin(angle)
    zero = torch.zeros((), device=scales.device)
    one = torch.ones((), device=scales.device)
    rotation = torch.stack([
        torch.stack([c, -s, zero]),
        torch.stack([s, c, zero]),
        torch.stack([zero, zero, one]),
    ])
    return rotation @ torch.diag(scales.square()) @ rotation.T

def project_covariance(mean_camera, covariance):
    x, y, z = mean_camera
    J = torch.stack([
        torch.stack([torch.tensor(fx, device=device) / z, torch.tensor(0.0, device=device), -fx * x / z.square()]),
        torch.stack([torch.tensor(0.0, device=device), torch.tensor(fy, device=device) / z, -fy * y / z.square()]),
    ])
    covariance_camera = R_camera @ covariance @ R_camera.T
    return J @ covariance_camera @ J.T + 1e-4 * torch.eye(2, device=device)

scales = torch.tensor([0.12, 0.08, 0.16], device=device)
mean = torch.tensor([0.0, 0.0, 2.0], device=device)
cov3 = covariance_3d(scales, torch.tensor(0.3, device=device))
cov2 = project_covariance(mean, cov3)
print("screen covariance:", cov2)


## 4. Degree-1 spherical harmonics view-dependent color

A 3DGS primitive carries view-dependent appearance rather than one fixed RGB. The degree-1 basis is enough to keep the actual directional-color path executable at small scale.


In [ ]:
def sh_degree1(direction):
    direction = F.normalize(direction, dim=-1)
    x, y, z = direction.unbind(-1)
    c0 = 0.2820947918
    c1 = 0.4886025119
    return torch.stack([
        torch.full_like(x, c0),
        -c1 * y,
        c1 * z,
        -c1 * x,
    ], dim=-1)

def evaluate_sh(coefficients, direction):
    basis = sh_degree1(direction)
    return torch.sigmoid(torch.einsum("...k,...kc->...c", basis, coefficients))

coeff = torch.randn(4, 3, device=device) * 0.2
view_color = evaluate_sh(coeff, torch.tensor([0.1, 0.2, 1.0], device=device))
print("SH color:", view_color)


## 5. Multi-Gaussian screen rasterizer

Four anisotropic Gaussians are projected onto a real 16x16 pixel grid, sorted by depth, converted to per-pixel alpha using the projected ellipse, and composited front-to-back.


In [ ]:
means = torch.tensor([
    [-0.18, -0.12, 1.8],
    [0.15, -0.05, 2.1],
    [-0.05, 0.18, 2.4],
    [0.20, 0.16, 2.8],
], device=device)
scales = torch.tensor([
    [0.10, 0.06, 0.12],
    [0.08, 0.11, 0.10],
    [0.12, 0.07, 0.09],
    [0.07, 0.07, 0.13],
], device=device)
angles = torch.tensor([0.0, 0.5, -0.3, 0.8], device=device)
opacities = torch.tensor([0.75, 0.65, 0.70, 0.55], device=device)
sh_coefficients = torch.randn(4, 4, 3, device=device) * 0.25

height = width = 16
y_grid, x_grid = torch.meshgrid(
    torch.arange(height, device=device, dtype=torch.float32),
    torch.arange(width, device=device, dtype=torch.float32),
    indexing="ij",
)
pixels = torch.stack([x_grid, y_grid], dim=-1)

camera_means, screen_means = project_points(means)
alpha_maps = []
color_values = []

for index in range(means.size(0)):
    cov3 = covariance_3d(scales[index], angles[index])
    cov2 = project_covariance(camera_means[index], cov3)
    inverse = torch.linalg.inv(cov2)
    offset = pixels - screen_means[index]
    mahalanobis = torch.einsum("...i,ij,...j->...", offset, inverse, offset)
    gaussian = torch.exp(-0.5 * mahalanobis)
    alpha_maps.append(opacities[index] * gaussian)

    view_direction = -camera_means[index]
    color_values.append(evaluate_sh(sh_coefficients[index], view_direction))

alpha_maps = torch.stack(alpha_maps)
color_values = torch.stack(color_values)
order = camera_means[:, 2].argsort()
alpha_maps = alpha_maps[order]
color_values = color_values[order]

image = torch.zeros(height, width, 3, device=device)
transmittance = torch.ones(height, width, device=device)
for index in range(alpha_maps.size(0)):
    alpha_i = alpha_maps[index].clamp(0, 0.99)
    weight_i = transmittance * alpha_i
    image = image + weight_i[..., None] * color_values[index]
    transmittance = transmittance * (1 - alpha_i)

print("rendered image:", image.shape)
print("mean RGB:", image.mean(dim=(0, 1)))
print("remaining transmittance:", transmittance.mean().item())


## References and provenance

- NeRF: density-to-alpha, transmittance and weighted color integration.
- 3D Gaussian Splatting: anisotropic 3D covariance, projection Jacobian, screen-space ellipse, spherical-harmonic appearance and depth-ordered alpha compositing.
